# H-05: Consistency & Predictability (Variance Stability)

**Question:** Can you rely on this agent to behave the same way every time — or is it fast one minute and slow the next?

| | |
|---|---|
| **H₀ (Null):** | Variance is equal across all fault categories. |
| **Hₐ (Alternative):** | At least one category has significantly higher variance (erratic behavior). |
| **Certification rule:** | Categories with CV ≥ 0.30 are flagged as "unstable". |
| **Metrics:** | time_to_detect, time_to_mitigate, reasoning_quality_score, hallucination_score |
| **Methods:** | Levene’s test (variance equality) + Coefficient of Variation (CV) |

### CV Thresholds
| CV Range | Label | Meaning |
|----------|-------|---------|
| CV < 0.15 | ✅ Stable | Agent behaves predictably |
| 0.15 ≤ CV < 0.30 | ⚠️ Moderate | Some variability, acceptable for most SRE use cases |
| CV ≥ 0.30 | ❌ Unstable | Erratic — cannot be relied on |

### Why CV (not just std)?
Standard deviation depends on scale. TTD std=50s is great for a 300s mean (CV=0.17) but terrible for a 100s mean (CV=0.50). CV normalizes by the mean, making it comparable across metrics.

### Configuration Note
Metric selection for this hypothesis is configurable per use case. Optional normalization may be applied when needed to align scales across metrics and fault categories.


In [1]:
import sys, os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True, source="quantitative"):
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        if source == "quantitative":
            val = q.get(metric_name)
        else:
            val = run["qualitative"].get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r['quantitative'].get('fault_detected') == 'Yes')
    faults = sorted(set(r['fault_name'] for r in runs))
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in faults:
        fn_det = sum(1 for r in runs if r['fault_name'] == fn and r['quantitative'].get('fault_detected') == 'Yes')
        print(f"    {fn}: {fn_det} detected")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 28 detected
    pod-delete: 23 detected
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 13 detected
    pod-network-corruption: 14 detected
    pod-network-loss: 12 detected
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 25 detected
    pod-cpu-hog: 25 detected
    pod-memory-hog: 20 detected


---
## Step 1: Per Sub-Fault Variance Overview

Shows how variable each sub-fault is. High CV = erratic behavior.


In [2]:
from scipy.stats import trim_mean

print("time_to_detect: variance per sub-fault (detected-only)")
print("=" * 85)
print(f"{'Category / Sub-Fault':30s} {'n':>4s} {'Mean':>8s} {'Std':>8s} {'CV':>7s} {'Flag':>10s}")
print("-" * 85)

for cat, runs in categories.items():
    sf_data = build_subfault_data(runs, "time_to_detect")
    all_vals = [v for vals in sf_data.values() for v in vals]
    pooled = np.array(all_vals)
    p_cv = np.std(pooled, ddof=1) / np.mean(pooled) if np.mean(pooled) != 0 else 0
    flag = '\u2705 stable' if p_cv < 0.15 else '\u26a0\ufe0f moderate' if p_cv < 0.30 else '\u274c UNSTABLE'
    print(f"  {cat:28s} {len(pooled):4d} {np.mean(pooled):7.1f}s {np.std(pooled, ddof=1):7.1f}s {p_cv:6.2f}  {flag}")
    for fn, vals in sorted(sf_data.items()):
        arr = np.array(vals)
        cv = np.std(arr, ddof=1) / np.mean(arr) if np.mean(arr) != 0 else 0
        sflag = '\u2705' if cv < 0.15 else '\u26a0\ufe0f' if cv < 0.30 else '\u274c'
        print(f"    {fn:26s} {len(arr):4d} {np.mean(arr):7.1f}s {np.std(arr, ddof=1):7.1f}s {cv:6.2f}  {sflag}")
    print()


time_to_detect: variance per sub-fault (detected-only)
Category / Sub-Fault              n     Mean      Std      CV       Flag
-------------------------------------------------------------------------------------
  application_fault              51   127.5s    31.1s   0.24  ⚠️ moderate
    container-kill               28   123.4s    25.1s   0.20  ⚠️
    pod-delete                   23   132.6s    37.1s   0.28  ⚠️

  network_fault                  39   233.7s   271.9s   1.16  ❌ UNSTABLE
    pod-dns-error                13   190.6s   259.5s   1.36  ❌
    pod-network-corruption       14   310.5s   323.7s   1.04  ❌
    pod-network-loss             12   190.7s   215.4s   1.13  ❌

  resource_fault                 70   241.4s    67.5s   0.28  ⚠️ moderate
    disk-fill                    25   216.9s    45.1s   0.21  ⚠️
    pod-cpu-hog                  25   234.7s    66.7s   0.28  ⚠️
    pod-memory-hog               20   280.3s    76.8s   0.27  ⚠️



---
## Step 2: Run H-05 — time_to_detect

Levene’s test checks if variances are equal. CV per category measures predictability.


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h05_consistency_predictability import run_consistency_test

ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect")

h05_ttd = run_consistency_test(ttd_data, metric_name="time_to_detect")

print(f"H-05: {h05_ttd.hypothesis_name}")
print(f"Metric: {h05_ttd.metric_name}")
print(f"Overall: {h05_ttd.overall_assessment}")
print("=" * 85)

# Levene's test
print(f"\n--- Levene's Test ---")
print(f"  F = {h05_ttd.levene_statistic:.4f},  p = {h05_ttd.levene_p:.6f}")
print(f"  Variances equal: {h05_ttd.variances_equal}")

# Per-category CV
print(f"\n--- Per-Category CV ---")
for c in h05_ttd.per_category:
    flag_icon = '\u2705' if c.cv_flag == 'stable' else '\u26a0\ufe0f' if c.cv_flag == 'moderate' else '\u274c'
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Pooled: mean={c.pooled_mean:.1f}s  std={c.pooled_std:.1f}s  CV={c.pooled_cv:.2f}  {flag_icon} {c.cv_flag}")
    print(f"    Sub-fault CV range: {c.within_cv_range}")
    for sf in c.sub_faults:
        sf_icon = '\u2705' if sf.cv_flag == 'stable' else '\u26a0\ufe0f' if sf.cv_flag == 'moderate' else '\u274c'
        print(f"      {sf.fault_name:25s}: mean={sf.mean:7.1f}s  std={sf.std:6.1f}s  CV={sf.cv:.2f}  {sf_icon}")

if h05_ttd.unstable_categories:
    print(f"\n\u274c Unstable categories: {', '.join(h05_ttd.unstable_categories)}")

if h05_ttd.warnings:
    print(f"\nWarnings: {h05_ttd.warnings}")


H-05: Consistency & Predictability
Metric: time_to_detect
Overall: variance_instability_detected

--- Levene's Test ---
  F = 16.9828,  p = 0.000000
  Variances equal: False

--- Per-Category CV ---

  application_fault (n=51, 2 sub-faults):
    Pooled: mean=127.5s  std=31.1s  CV=0.24  ⚠️ moderate
    Sub-fault CV range: 0.20–0.28
      container-kill           : mean=  123.4s  std=  25.1s  CV=0.20  ⚠️
      pod-delete               : mean=  132.6s  std=  37.1s  CV=0.28  ⚠️

  network_fault (n=39, 3 sub-faults):
    Pooled: mean=233.7s  std=271.9s  CV=1.16  ❌ unstable
    Sub-fault CV range: 1.04–1.36
      pod-dns-error            : mean=  190.6s  std= 259.5s  CV=1.36  ❌
      pod-network-corruption   : mean=  310.5s  std= 323.7s  CV=1.04  ❌
      pod-network-loss         : mean=  190.7s  std= 215.4s  CV=1.13  ❌

  resource_fault (n=70, 3 sub-faults):
    Pooled: mean=241.3s  std=67.5s  CV=0.28  ⚠️ moderate
    Sub-fault CV range: 0.21–0.28
      pod-cpu-hog              : mean=  234.

---
## Step 3: Run H-05 — time_to_mitigate


In [4]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate")

h05_ttm = run_consistency_test(ttm_data, metric_name="time_to_mitigate")

print(f"H-05: {h05_ttm.metric_name}  |  {h05_ttm.overall_assessment}")
print(f"  Levene: F={h05_ttm.levene_statistic:.4f}, p={h05_ttm.levene_p:.6f}, equal={h05_ttm.variances_equal}")
print("=" * 85)
for c in h05_ttm.per_category:
    flag_icon = '\u2705' if c.cv_flag == 'stable' else '\u26a0\ufe0f' if c.cv_flag == 'moderate' else '\u274c'
    print(f"  {c.category:20s}: mean={c.pooled_mean:7.1f}s  std={c.pooled_std:6.1f}s  CV={c.pooled_cv:.2f}  {flag_icon} {c.cv_flag}")
if h05_ttm.unstable_categories:
    print(f"  \u274c Unstable: {', '.join(h05_ttm.unstable_categories)}")


H-05: time_to_mitigate  |  variance_instability_detected
  Levene: F=25.0522, p=0.000000, equal=False
  application_fault   : mean=  314.0s  std=  70.8s  CV=0.23  ⚠️ moderate
  network_fault       : mean=  511.1s  std= 327.2s  CV=0.64  ❌ unstable
  resource_fault      : mean=  502.9s  std= 155.8s  CV=0.31  ❌ unstable
  ❌ Unstable: network_fault, resource_fault


---
## Step 4: Run H-05 — reasoning_quality_score


In [5]:
rsn_data = {}
for cat, runs in categories.items():
    rsn_data[cat] = build_subfault_data(runs, "reasoning_quality_score",
                                        detected_only=False, source="qualitative")

h05_rsn = run_consistency_test(rsn_data, metric_name="reasoning_quality_score")

print(f"H-05: {h05_rsn.metric_name}  |  {h05_rsn.overall_assessment}")
print(f"  Levene: F={h05_rsn.levene_statistic:.4f}, p={h05_rsn.levene_p:.6f}, equal={h05_rsn.variances_equal}")
print("=" * 85)
for c in h05_rsn.per_category:
    flag_icon = '\u2705' if c.cv_flag == 'stable' else '\u26a0\ufe0f' if c.cv_flag == 'moderate' else '\u274c'
    print(f"  {c.category:20s}: mean={c.pooled_mean:.2f}/10  std={c.pooled_std:.2f}  CV={c.pooled_cv:.2f}  {flag_icon} {c.cv_flag}")
if h05_rsn.unstable_categories:
    print(f"  \u274c Unstable: {', '.join(h05_rsn.unstable_categories)}")


H-05: reasoning_quality_score  |  variance_instability_detected
  Levene: F=0.0462, p=0.954826, equal=True
  application_fault   : mean=7.78/10  std=2.26  CV=0.29  ⚠️ moderate
  network_fault       : mean=4.41/10  std=1.68  CV=0.38  ❌ unstable
  resource_fault      : mean=6.38/10  std=1.82  CV=0.28  ⚠️ moderate
  ❌ Unstable: network_fault


---
## Step 5: Run H-05 — hallucination_score


In [6]:
hal_data = {}
for cat, runs in categories.items():
    hal_data[cat] = build_subfault_data(runs, "hallucination_score",
                                        detected_only=False, source="qualitative")

h05_hal = run_consistency_test(hal_data, metric_name="hallucination_score")

print(f"H-05: {h05_hal.metric_name}  |  {h05_hal.overall_assessment}")
print(f"  Levene: F={h05_hal.levene_statistic:.4f}, p={h05_hal.levene_p:.6f}, equal={h05_hal.variances_equal}")
print("=" * 85)
for c in h05_hal.per_category:
    flag_icon = '\u2705' if c.cv_flag == 'stable' else '\u26a0\ufe0f' if c.cv_flag == 'moderate' else '\u274c'
    print(f"  {c.category:20s}: mean={c.pooled_mean:.3f}  std={c.pooled_std:.3f}  CV={c.pooled_cv:.2f}  {flag_icon} {c.cv_flag}")
if h05_hal.unstable_categories:
    print(f"  \u274c Unstable: {', '.join(h05_hal.unstable_categories)}")


H-05: hallucination_score  |  variance_instability_detected
  Levene: F=9.5488, p=0.000103, equal=False
  application_fault   : mean=0.080  std=0.090  CV=1.14  ❌ unstable
  network_fault       : mean=0.230  std=0.130  CV=0.56  ❌ unstable
  resource_fault      : mean=0.130  std=0.110  CV=0.87  ❌ unstable
  ❌ Unstable: application_fault, network_fault, resource_fault


---
## Step 6: Summary & Interpretation


In [7]:
print("H-05 Consistency & Predictability \u2014 Summary")
print("=" * 85)

results = {
    "time_to_detect": h05_ttd,
    "time_to_mitigate": h05_ttm,
    "reasoning_quality_score": h05_rsn,
    "hallucination_score": h05_hal,
}

for metric, r in results.items():
    lev_lbl = "\u2705 EQUAL" if r.variances_equal else "\u274c UNEQUAL"
    print(f"\n  {metric}:")
    print(f"    Levene: {lev_lbl}  (F={r.levene_statistic:.2f}, p={r.levene_p:.4f})")
    print(f"    CV: ", end="")
    parts = []
    for cat, cv in r.cv_per_category.items():
        flag = r.cv_flags[cat]
        icon = '\u2705' if flag == 'stable' else '\u26a0\ufe0f' if flag == 'moderate' else '\u274c'
        parts.append(f"{cat.replace('_fault','')}={cv:.2f} {icon}")
    print("  ".join(parts))
    if r.unstable_categories:
        print(f"    \u274c Unstable: {', '.join(r.unstable_categories)}")

print("\n" + "=" * 85)
print("\nInterpretation:")
print("  - CV < 0.15: Agent responds consistently. Operational plans can rely on typical values.")
print("  - CV 0.15-0.30: Moderate variability. Use CI bounds (H-01) rather than point estimates.")
print("  - CV \u2265 0.30: UNSTABLE. Agent behavior is unpredictable for this category/metric.")
print("    Consider: wider safety margins, fallback mechanisms, or targeted retraining.")
print("  - Levene UNEQUAL: The agent's reliability is not consistent across categories.")
print("    Some categories are predictable while others are erratic.")


H-05 Consistency & Predictability — Summary

  time_to_detect:
    Levene: ❌ UNEQUAL  (F=16.98, p=0.0000)
    CV: application=0.24 ⚠️  network=1.16 ❌  resource=0.28 ⚠️
    ❌ Unstable: network_fault

  time_to_mitigate:
    Levene: ❌ UNEQUAL  (F=25.05, p=0.0000)
    CV: application=0.23 ⚠️  network=0.64 ❌  resource=0.31 ❌
    ❌ Unstable: network_fault, resource_fault

  reasoning_quality_score:
    Levene: ✅ EQUAL  (F=0.05, p=0.9548)
    CV: application=0.29 ⚠️  network=0.38 ❌  resource=0.28 ⚠️
    ❌ Unstable: network_fault

  hallucination_score:
    Levene: ❌ UNEQUAL  (F=9.55, p=0.0001)
    CV: application=1.14 ❌  network=0.56 ❌  resource=0.87 ❌
    ❌ Unstable: application_fault, network_fault, resource_fault


Interpretation:
  - CV < 0.15: Agent responds consistently. Operational plans can rely on typical values.
  - CV 0.15-0.30: Moderate variability. Use CI bounds (H-01) rather than point estimates.
  - CV ≥ 0.30: UNSTABLE. Agent behavior is unpredictable for this category/metric.

---
## Step 7: JSON Output


In [8]:
output = {
    "hypothesis_id": "H-05",
    "hypothesis_name": "Consistency & Predictability (Variance Stability)",
    "null_hypothesis": "Variance is equal across all fault categories.",
    "alt_hypothesis": "At least one category has significantly higher variance.",
    "certification_rule": "Categories with CV >= 0.30 are flagged as unstable.",
    "hypothesis_metadata": {
        "method": "Levene's test (median-based) + Coefficient of Variation",
        "cv_thresholds": {"stable": "<0.15", "moderate": "0.15-0.30", "unstable": ">=0.30"},
        "aggregation": "pooled per-run values within each category",
        "alpha": 0.05,
        "categories": list(categories.keys()),
        "mode": "both (always active)",
        "configuration_note": "Metric selection is configurable. Optional normalization available.",
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "levene": {
            "statistic": r.levene_statistic,
            "p_value": r.levene_p,
            "variances_equal": r.variances_equal,
        },
        "per_category": [
            {
                "category": c.category,
                "n": c.n,
                "pooled_mean": c.pooled_mean,
                "pooled_std": c.pooled_std,
                "cv": c.pooled_cv,
                "cv_flag": c.cv_flag,
                "within_cv_range": c.within_cv_range,
            }
            for c in r.per_category
        ],
        "unstable_categories": r.unstable_categories,
    }

print(json.dumps(output, indent=2))


{
  "hypothesis_id": "H-05",
  "hypothesis_name": "Consistency & Predictability (Variance Stability)",
  "null_hypothesis": "Variance is equal across all fault categories.",
  "alt_hypothesis": "At least one category has significantly higher variance.",
  "certification_rule": "Categories with CV >= 0.30 are flagged as unstable.",
  "hypothesis_metadata": {
    "method": "Levene's test (median-based) + Coefficient of Variation",
    "cv_thresholds": {
      "stable": "<0.15",
      "moderate": "0.15-0.30",
      "unstable": ">=0.30"
    },
    "aggregation": "pooled per-run values within each category",
    "alpha": 0.05,
    "categories": [
      "application_fault",
      "network_fault",
      "resource_fault"
    ],
    "mode": "both (always active)",
    "configuration_note": "Metric selection is configurable. Optional normalization available."
  },
  "results": {
    "time_to_detect": {
      "overall_assessment": "variance_instability_detected",
      "levene": {
        "statis